<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/YapayZekaninPrensipleriFinal_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sayın Hocam,

Ödevi incelerken aşağıdaki hususları dikkate almanızı rica ederim.
Hibrit İş Akışı Tasarımı ve LLM Prompt Mühendisliği ödevi kapsamında yaptığım çalışma hakkında kısa bir bilgilendirme paylaşmak isterim.

Ödevde yer alan Yapay Zeka modeli entegrasyonu aşamasında yerel ortamımda bir LLM modeli çalıştırmayı denedim. Ayrıca Google API Key kullanarak dış servis üzerinden model çağırma girişiminde de bulundum; ancak teknik erişim ve yetkilendirme sorunları nedeniyle modeli aktif olarak çalıştıramadım.

Bu sebeple çalışmanın sürekliliğini sağlayabilmek adına, yapay zeka aşamasını teorik olarak tasarlayıp pratik bölümde Regex katmanını daha geniş kapsamlı ele aldım. Regex aşamasında anahtar kelime sınıflandırmasını manuel gözlem ve senaryo bazlı analiz ile genişleterek, belirsiz vakaları en aza indirmeye çalıştım. Yani AI modelinin üstleneceği sınıflandırma yükünü kısmen Regex ve anahtar kelime mantığı ile simüle etmeye çalıştım.

Bu yaklaşım ile iş akışının mantıksal bütünlüğünü korumayı ve sistem tasarımını kavramsal olarak doğru şekilde yansıtmayı hedefledim. Teknik model entegrasyonunu gerçekleştirememiş olsam da süreç tasarımını ve kategori belirleme mantığını eksiksiz göstermeye özen gösterdim.


Bilgilerinize sunarım.

Saygılarımla

##   Excel Dosyasının Yüklenmesi

Sağlanan Excel dosyası 'v1_f_IT_Department_Sorunlar_.xlsx' bir pandas DataFrame'ine yüklenmesi


In [ ]:
import pandas as pd

df = pd.read_excel('/content/v1_f_IT_Department_Sorunlar_.xlsx')

print("DataFrame loaded successfully. Displaying the first 5 rows:")
print(df.head())

## Veri İncelenmesi


In [ ]:
print("Column names:")
print(df.columns)

print("\nDataFrame information (data types and non-null counts):")
df.info()

## Aşama 1: Regex ile Ön Sınıflandırma (Hata-İstek-Diger)


In [ ]:
import re

# 'HATA' kategorisi için anahtar kelimeler
hata_keywords = r'sorun|hata|çalışmıyor|ulaşılamıyor|problem|eror|açılmıyor|bozuk|kesildi|kayboldu|yanıt vermiyor|bağlantı kurulamıyor|geçersiz|kilitlendi|dondu|yavaş|hatası|düşüyor|kopukluk|yüklenmiyor|aksıyor|bekletiyor'

# 'İSTEK' kategorisi için anahtar kelimeler
istek_keywords = r'talep|istek|ekleme|kurulum|tanımlama|ihtiyacım var|yetki'

# Yeni bir sütun oluştur ve varsayılan olarak 'DİĞER' ata
df['pre_classification'] = 'DİGER'

# 'HATA' olarak sınıflandırılacak satırları belirle
is_hata_condition = df['body'].str.contains(hata_keywords, case=False, na=False)
df.loc[is_hata_condition, 'pre_classification'] = 'HATA'

# 'İSTEK' olarak sınıflandırılacak satırları belirle
is_not_hata = (df['pre_classification'] != 'HATA')
is_istek_condition = df['body'].str.contains(istek_keywords, case=False, na=False)
df.loc[is_not_hata & is_istek_condition, 'pre_classification'] = 'İSTEK'

print("Ön sınıflandırma tamamlandı. Kategorilerin dağılımı:")
print(df['pre_classification'].value_counts())

print("\nÖn sınıflandırma yapılmış ilk 5 satır:")
print(df[['body', 'pre_classification']].head())

## Veri Yükleme ve İlk İnceleme Doğrulaması


In [ ]:
print(df['pre_classification'].value_counts())

## LLM ile Belirsiz Vakaları Sınıflandırma ve Güncelleme


In [ ]:
df_to_classify_with_llm = df[df['pre_classification'] == 'DİGER'].copy()

print("DataFrame 'df_to_classify_with_llm' created successfully. Displaying the first 5 rows:")
print(df_to_classify_with_llm.head())

In [ ]:
llm_classification_prompt = """
Aşağıdaki IT destek talebi metnini dikkatlice analiz edin ve en uygun kategoriye sınıflandırın. Sadece atanacak kategori adını döndürün.

Kategoriler ve Tanımları:
- Bağlantı: Ağ sorunları, internet erişim problemleri, VPN bağlantı sorunları, Wi-Fi bağlantısı kesilmesi vb.
- Donanım: Bilgisayar, monitör, klavye, fare, yazıcı, tarayıcı, telefon gibi fiziksel cihazlarla ilgili arızalar veya talepler.
- Erişim: Yazılımlara, sistemlere (ERP, CRM), sunuculara veya belirli kaynaklara erişim sorunları, parola sıfırlama, yetkilendirme vb.
- Yazılım: İşletim sistemi, Office programları, özel iş uygulamaları (SAP, İK Portal), antivirüs yazılımları, driver'lar vb. ile ilgili hatalar, çakışmalar veya yeni kurulum talepleri.
- Kullanıcı Yönetimi: Kullanıcı hesabı açma, kapatma, şifre değiştirme, yetki verme, erişim rolleri tanımlama gibi kullanıcı hesapları ve yetkileriyle ilgili işlemler.
- Raporlama: Sistemlerden veri çekme, özel raporlar oluşturma, mevcut raporlarda hata veya değişiklik talepleri gibi raporlama ihtiyaçları.
- Güvenlik: Virüs, kötü amaçlı yazılım tespiti, güvenlik duvarı sorunları, veri sızıntısı endişeleri, güvenlik politikası uygulamaları.
- Diğer: Yukarıdaki kategorilere uymayan, ancak IT departmanının ilgilendiği genel destek talepleri veya bilgi talepleri.

IT DESTEK TALEBİ METNİ:
{email_body}
"""

df_to_classify_with_llm['llm_classification'] = None

print("LLM classification prompt defined and 'llm_classification' column initialized.")

In [ ]:
pip install --upgrade google-generativeai

In [ ]:
import google.generativeai as genai
model = genai.GenerativeModel("gemini-1.5-flash")

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# API anahtarını buraya gir
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel("gemini-pro")
llm_classification_prompt = """
Aşağıdaki IT destek talebi metnini dikkatlice analiz edin ve en uygun kategoriye sınıflandırın. Sadece atanacak kategori adını döndürün.
Kategoriler ve Tanımları:
- Bağlantı: Ağ sorunları, internet erişim problemleri, VPN bağlantı sorunları, Wi-Fi bağlantısı kesilmesi vb.
- Donanım: Bilgisayar, monitör, klavye, fare, yazıcı, tarayıcı, telefon gibi fiziksel cihazlarla ilgili arızalar veya talepler.
- Erişim: Yazılımlara, sistemlere (ERP, CRM), sunuculara veya belirli kaynaklara erişim sorunları, parola sıfırlama, yetkilendirme vb.
- Yazılım: İşletim sistemi, Office programları, özel iş uygulamaları (SAP, İK Portal), antivirüs yazılımları, driver'lar vb. ile ilgili hatalar, çakışmalar veya yeni kurulum talepleri.
- Kullanıcı Yönetimi: Kullanıcı hesabı açma, kapatma, şifre değiştirme, yetki verme, erişim rolleri tanımlama gibi kullanıcı hesapları ve yetkileriyle ilgili işlemler.
- Raporlama: Sistemlerden veri çekme, özel raporlar oluşturma, mevcut raporlarda hata veya değişiklik talepleri gibi raporlama ihtiyaçları.
- Güvenlik: Virüs, kötü amaçlı yazılım tespiti, güvenlik duvarı sorunları, veri sızıntısı endişeleri, güvenlik politikası uygulamaları.
- Diğer: Yukarıdaki kategorilere uymayan, ancak IT departmanının ilgilendiği genel destek talepleri veya bilgi talepleri.

IT DESTEK TALEBİ METNİ:
{email_body}
"""

df_to_classify_with_llm['llm_classification'] = None

for index, row in df_to_classify_with_llm.iterrows():
    email_body_text = row['body']
    formatted_prompt = llm_classification_prompt.format(email_body=email_body_text)
    try:
        response = gemini_model.generate_content(formatted_prompt)
        llm_result = response.text.strip()
        df_to_classify_with_llm.loc[index, 'llm_classification'] = llm_result
    except Exception as e:
        print(f"Error classifying row {index}: {e}")
        df_to_classify_with_llm.loc[index, 'llm_classification'] = 'LLM_ERROR'

print("LLM classification for 'DİGER' entries completed.")
print(df_to_classify_with_llm['llm_classification'].value_counts())
print("\nFirst 5 rows of df_to_classify_with_llm with LLM classification:")
print(df_to_classify_with_llm[['body', 'pre_classification', 'llm_classification']].head())

In [ ]:
df.loc[df_to_classify_with_llm.index, 'pre_classification'] = df_to_classify_with_llm['llm_classification']

print("LLM sınıflandırma sonuçları ana DataFrame'e başarıyla entegre edildi.")
print("Güncellenmiş 'pre_classification' sütununun değer dağılımı:")
print(df['pre_classification'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='pre_classification', order=df['pre_classification'].value_counts().index, palette='viridis')
plt.title('Final Dağılım - IT Destek Talepleri Kategorileri')
plt.xlabel('Kategori')
plt.ylabel('Sayı')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Final category distribution bar chart displayed.")